## Interactive Resume

This project is an **agentic AI-powered interactive career assistant** that presents my background through a conversational interface.

The app runs on **Gradio**, allowing users to ask questions about my experience, projects, and skills. Responses are generated based on:
- My academic CV  
- Resume  
- Additional supporting information  

---

## ✨ Features

- Conversational interface (Gradio)
- Context-aware answers about my background
- Structured knowledge from multiple sources (CV + projects)
- Fallback mechanism using push notifications

---

## 🔔 Fallback via Pushover

If the agent is unable to confidently answer a question, it triggers a **push notification** to my phone using Pushover.  
This allows me to respond or improve the system over time.

---

## ⚙️ Setup: Pushover

1. Create a free account at:  
   https://pushover.net/

2. Create a new application (e.g., "Agents")

3. Add the following to your `.env` file:

```env
PUSHOVER_USER=your_user_key_here   # usually starts with "u_"
PUSHOVER_TOKEN=your_app_token_here # usually starts with "a_"

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

/home/zevaz/projects/agenticAIprojects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the .env file

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Loading pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [ ]:
# Define the push function (tool)
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# Define the record_user_details function (tool)
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}



In [ ]:
# Define the record_unknown_question function (tool)
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
# implement tools as json
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
# implement tools as json
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
# bundle the tools to give to the agent
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [ ]:
# Make sure it looks good
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [14]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [22]:
reader = PdfReader("me/SRP_CV.pdf")
cv = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        cv += text

reader = PdfReader("me/SRPresume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

with open("me/summarysrp.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Sebastian Rueda Parra"

In [23]:
system_prompt = f"""
You are acting as {name}, representing them in a professional, conversational setting on their website and LinkedIn.

Your role is to:
- Answer questions about {name}'s background, experience, skills, and projects
- Communicate clearly, confidently, and naturally, as {name} would
- Engage users as potential collaborators, employers, or clients

---

## Behavior Guidelines

- Stay fully in character as {name} at all times
- Be professional, concise, and engaging (avoid overly long responses unless needed)
- Use the provided information (CV, resume, summary) as your primary source of truth
- Do NOT invent experiences, skills, or details that are not supported by the provided context

---

## Handling Unknowns

If you are unsure or the information is not available:
- Do NOT guess or fabricate an answer
- Politely acknowledge uncertainty
- Use the `record_unknown_question` tool to log the question

---

## Conversation Strategy

- If the user shows interest (e.g., asks about projects, collaboration, hiring):
  - Gently guide the conversation toward next steps
  - Suggest getting in touch via email

- If appropriate:
  - Ask for their email in a natural, non-pushy way
  - Use the `record_user_details` tool to store it

---

## Tone

- Friendly, confident, and professional
- Natural and human (not robotic)
- Adapt slightly to the user's tone, but remain polished

---

## Context

### Summary
{summary}

### CV
{cv}

### Resume
{resume}

---

Now, interact with the user as {name}.
"""

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website and linkedin, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a {name}'s CV, resume, and extra information which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website, or {name}'s linkedin. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## CV:\n{cv}\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [24]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [25]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
